#Initialization

In [0]:
%sql
USE CATALOG workspace

# Read Bronze table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df AS
SELECT * FROM workspace.bronze.erp_px_cat_g1v2

#Silver Transformations

##Trimming

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_trimmed AS
SELECT TRIM(id)          AS id,
       TRIM(cat)         AS cat,
       TRIM(subcat)      AS subcat,
       TRIM(maintenance) AS maintenance
FROM df

## Normalize Maintenance Flag to Boolean

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_maint AS
SELECT id, cat, subcat,
       CASE WHEN UPPER(maintenance) = 'YES' THEN TRUE
            WHEN UPPER(maintenance) = 'NO'  THEN FALSE
            ELSE NULL END AS maintenance
FROM df_trimmed

## Renaming Columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_renamed AS
SELECT id          AS category_id,
       cat         AS category,
       subcat      AS subcategory,
       maintenance AS maintenance_flag
FROM df_maint

## Sanity checks of dataframe

In [0]:
%sql
SELECT * FROM df_renamed LIMIT 10

#Writing Silver Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.erp_product_category USING DELTA AS
SELECT * FROM df_renamed

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_product_category LIMIT 10